# prepare_mami3 — CORREGIDO

**Cambios principales respecto al original:**
- **Cell 3 (extract_physio_features):** Eliminada la normalización por sujeto individual (RobustScaler + Z-score por sujeto). Ahora extrae features *raw* (solo limpieza de NaN/Inf). La normalización global se aplica en Cell 7b después del split.
- **Cell 7b (NUEVO):** `GlobalPhysioNormalizer` — fit sobre train, transform sobre train+val. Evita data leakage y preserva las diferencias de magnitud entre memes sexistas y no sexistas que el modelo necesita aprender.
- Cells 0,1,2,4,5,6,8 sin cambios.

In [1]:
import json

with open("../data/dataset_task3_exist2026/training.json") as f:
    data = json.load(f)

In [2]:
with open("../data/dataset_task3_exist2026/test.json") as f:
    test_data = json.load(f)
 
with open("../data/EXIST 2026 Videos Dataset/training/EXIST2026_training.json") as f:
# with open("../data/EXIST 2026 Videos Dataset/training/metadata.json") as f:
    test_gold = json.load(f)

with open("../data/EXIST 2026 Videos Dataset/training/metadata.json") as f:
    metadata = json.load(f)

gold_index = {
    str(v["id_EXIST"]): v["labels_task3_1"]
    for v in test_gold.values()
}

extra_sounds_by_tiktok = {
    str(v.get("id_Tiktok")): v.get("extra_sounds", {})
    for v in metadata.values()
    if v.get("id_Tiktok") is not None
}

extra_sounds_by_exist = {
    str(v.get("id_EXIST")): v.get("extra_sounds", {})
    for v in metadata.values()
    if v.get("id_EXIST") is not None
}

In [3]:
import numpy as np

# ── CORREGIDO ────────────────────────────────────────────────────────────────
# ANTES: se aplicaba RobustScaler + Z-score POR CADA SUJETO individualmente.
# PROBLEMA: normalizar por sujeto garantiza que todos los usuarios acaban con
#   media=0 y std=1, eliminando exactamente las diferencias de magnitud entre
#   memes sexistas y no sexistas (más fixations, más RT, diferente EEG power)
#   que el paper detectó como significativas (p<0.001).
# FIX: aquí solo limpiamos NaN/Inf. La normalización global (ajustada sobre
#   TODO el train set) se aplica en la Cell 7b, después del split.
# ─────────────────────────────────────────────────────────────────────────────

def extract_physio_features_raw(sensorial):
    """
    Extrae features fisiológicas sin normalizar.
    Solo limpia NaN/Inf para evitar errores numéricos en pasos posteriores.
    Devuelve: {"ET": [[vec_suj1], ...], "HR": [...], "EEG": [...]}
    """
    output = {}

    for modality in ["ET", "HR", "EEG"]:
        if modality not in sensorial.get("modalities", {}):
            continue

        users_data = sensorial["modalities"][modality]["by_user"]
        subject_vectors = []

        for user_id, features_dict in users_data.items():
            vec = np.array(
                [v if v is not None else np.nan for v in features_dict.values()],
                dtype=float
            )
            # Limpieza básica de NaN/Inf — sin normalizar
            if np.all(np.isnan(vec)):
                vec = np.zeros_like(vec)
            else:
                mean_val = np.nanmean(vec)
                vec = np.nan_to_num(vec, nan=mean_val, posinf=mean_val, neginf=mean_val)

            subject_vectors.append(vec.tolist())

        output[modality] = subject_vectors

    return output


In [4]:
import json

CACHE_FILE = "../data/EXIST 2026 Videos Dataset/training/qwen_verdicts_alc3.json"

with open(CACHE_FILE) as f:
    verdict_cache = json.load(f)

def get_verdict_from_cache(tiktok_id: str) -> str:
    return verdict_cache.get(str(tiktok_id), "")

for meme_id, sample in data.items():
    tiktok_id = sample.get("id_Tiktok")
    sample["qwen_verdict"] = get_verdict_from_cache(tiktok_id)

print(f"✅ Verdicts cargados desde caché para {len(data)} memes.")

# CACHE_FILE_2 = "../data/EXIST 2026 Videos Dataset/training/qwen_verdicts_alc3_redo.json"

# with open(CACHE_FILE_2) as f:
#     verdict_cache = json.load(f)

# def get_verdict_from_cache(tiktok_id: str) -> str:
#     return verdict_cache.get(str(tiktok_id), "")

# for meme_id, sample in data.items():
#     tiktok_id = sample.get("id_Tiktok")
#     sample["qwen_verdict_2"] = get_verdict_from_cache(tiktok_id)

# print(f"✅ Verdicts cargados desde caché para {len(data)} memes.")

✅ Verdicts cargados desde caché para 2006 memes.


In [5]:
for meme_id, sample in data.items():
    print(sample["qwen_verdict"])
    # print(sample["qwen_verdict_2"])

TRANSCRIPTION: ¡Red de caja! ¡Altor, el Rayman ha conseguido impedir el tiro de la supertécnica!
OCR: *Cuando ves a las del 08 en la fiesta*
Tu amigo el menos salido:
REASONING: The image shows a scene where one character is being pushed off a ledge by another. The OCR text suggests a scenario where someone sees attractive girls at a party, and the least outgoing friend reacts dramatically. This combination implies that the friend's reaction is due to his inability to interact with these girls, reinforcing a stereotype about men's behavior around attractive women. However, the image itself does not explicitly degrade women or portray them as inferior.
LABEL: NO this video is not sexist

TRANSCRIPTION: A woman walks alone at dawn, and automatically, the speaker feels the need to protect her.
OCR: MUJER: *camina sola de madrugada* YO AUTOMATICAMENTE: OVER SPEED ETERNIUM LEVELS CRITICAL IX. M. Y IS EXCESIVE VELOCITY
REASONING: The visual frames depict a woman walking alone at dawn, which 

In [6]:
import re

def parse_qwen_verdict(output: str) -> dict:
    transcription_match = re.search(r'TRANSCRIPTION:\s*(.*?)\nOCR:', output, re.DOTALL)
    ocr_match = re.search(r'OCR:\s*(.*?)\nREASONING:', output, re.DOTALL)
    reasoning_match = re.search(r'REASONING:\s*(.*?)\nLABEL:', output, re.DOTALL)
    label_match = re.search(r'LABEL:\s*(YES.*?|NO.*?)\n?', output)

    return {
        "transcription": transcription_match.group(1).strip() if transcription_match else "",
        "ocr": ocr_match.group(1).strip() if ocr_match else "",
        "reasoning": reasoning_match.group(1).strip() if reasoning_match else "",
        "label": label_match.group(1).strip() if label_match else "",
    }

In [7]:
def prepare_labels(ls):
    aux = []
    for l in ls:
        if l == "YES":
            aux.append(1)
        else:
            aux.append(0)

    return aux

In [8]:
dataset = []

for meme_id, sample in data.items():
    id = int(sample["id_EXIST"])
    task1      = sample["labels_task3_1"]
    task1 = prepare_labels(task1)
    physio_raw = extract_physio_features_raw(sample["sensorial"])
    lang = sample.get("lang")
 
    qwen_output  = sample.get("qwen_verdict", "")
    parsed_qwen = parse_qwen_verdict(qwen_output)

    # qwen_output_2  = sample.get("qwen_verdict_2", "")
    # parsed_qwen_2 = parse_qwen_verdict(qwen_output_2)

    extra_sounds = extra_sounds_by_tiktok.get(
        str(sample.get("id_Tiktok")),
        extra_sounds_by_exist.get(str(sample.get("id_EXIST")), sample.get("extra_sounds", {}))
    )

    dataset.append({
        "id": id,
        "id_Tiktok": sample.get("id_Tiktok"),
        "physio": physio_raw,
        "label": task1,
        "lang": lang,
        "qwen_transcription": parsed_qwen["transcription"],
        "qwen_ocr": parsed_qwen["ocr"],
        "qwen_reasoning": parsed_qwen["reasoning"],
        "qwen_label": parsed_qwen["label"],
        "extra_sounds": extra_sounds
    })

In [9]:
import json

CACHE_FILE = "../data/EXIST 2026 Videos Dataset/training/qwen_verdicts_alc3.json"

with open(CACHE_FILE) as f:
    verdict_cache = json.load(f)

def get_verdict_from_cache(tiktok_id: str) -> str:
    return verdict_cache.get(str(tiktok_id), "")

for meme_id, sample in test_data.items():
    tiktok_id = sample.get("id_Tiktok")
    sample["qwen_verdict"] = get_verdict_from_cache(tiktok_id)

print(f"✅ Verdicts cargados desde caché para {len(data)} memes.")

# CACHE_FILE_2 = "../data/EXIST 2026 Videos Dataset/training/qwen_verdicts_alc3_redo.json"

# with open(CACHE_FILE_2) as f:
#     verdict_cache = json.load(f)

# def get_verdict_from_cache(tiktok_id: str) -> str:
#     return verdict_cache.get(str(tiktok_id), "")

# for meme_id, sample in test_data.items():
#     tiktok_id = sample.get("id_Tiktok")
#     sample["qwen_verdict_2"] = get_verdict_from_cache(tiktok_id)

# print(f"✅ Verdicts cargados desde caché para {len(data)} memes.")

✅ Verdicts cargados desde caché para 2006 memes.


In [10]:
test_dataset = []
 
for meme_id, sample in test_data.items():
    exist_id  = str(sample["id_EXIST"])
    tiktok_id = sample.get("id_Tiktok")
    lang = sample.get("lang")
 
    # Labels desde el gold (fallback a lista vacía si no se encuentra)
    raw_labels = gold_index.get(exist_id, [])
    labels     = prepare_labels(raw_labels)     # mismo helper que en training
 
    physio_raw  = extract_physio_features_raw(sample["sensorial"])
    qwen_output  = sample.get("qwen_verdict", "")
    parsed_qwen = parse_qwen_verdict(qwen_output)

    # qwen_output_2  = sample.get("qwen_verdict_2", "")
    # parsed_qwen_2 = parse_qwen_verdict(qwen_output_2)

    print(parsed_qwen)

    extra_sounds = extra_sounds_by_tiktok.get(
        str(tiktok_id),
        extra_sounds_by_exist.get(exist_id, sample.get("extra_sounds", {}))
    )
 
    test_dataset.append({
        "id":                 int(exist_id),
        "id_Tiktok": tiktok_id,
        "physio":             physio_raw,
        "label":              labels,
        "lang": lang,
        "qwen_transcription": parsed_qwen["transcription"],
        "qwen_ocr":           parsed_qwen["ocr"],
        "qwen_reasoning":     parsed_qwen["reasoning"],
        "qwen_label":         parsed_qwen["label"],
        "extra_sounds": extra_sounds
    })
 
print(f"Test samples construidos: {len(test_dataset)}")
 
# Verificación: cuántos samples tienen labels del gold
n_with_labels = sum(1 for s in test_dataset if s["label"])
print(f"  con labels del gold: {n_with_labels}/{len(test_dataset)}")

{'transcription': 'A mí me llevan preso como diga lo que opino de esto ahora mismo. A mí me llevan preso.', 'ocr': 'yo viendo que la gorda del salon tiene 10 en educación física y yo tengo 7:', 'reasoning': 'The video juxtaposes a humorous situation where one student expresses frustration about receiving a lower grade than a heavier classmate in physical education. The use of the term "gorda" (fat girl) and the implication that her weight should correlate negatively with her physical abilities perpetuates harmful stereotypes about body size and physical capability. This combination of visuals and text promotes a sexist and body-shaming message by mocking the student\'s weight.', 'label': 'YES'}
{'transcription': 'Cuando le dan patadas a una mamá embarazada, los bebecitos cuando le dan una patadita a la mamá embarazada no pasa nada. Ah, pero si yo voy y le doy una patada a la mamá, ahí se me echan todos en...', 'ocr': 'Yo el menos machista:', 'reasoning': 'The video features a person sp

In [11]:
import json
import random
random.seed(42)
import os
from scipy import stats
from sklearn.preprocessing import RobustScaler

# ── NUEVO: Normalización global por modalidad ─────────────────────────────────
# Por qué es necesario:
#   El paper detecta diferencias de magnitud entre memes sexistas y no sexistas:
#   más fixation count, mayor reaction time, diferente EEG power (p<0.001).
#   Si normalizamos por sujeto individualmente (como hacía el código original),
#   forzamos media=0 std=1 para cada usuario y destruimos esa señal.
#   La solución correcta es ajustar UN scaler global sobre todos los sujetos
#   del train set y aplicarlo igual a train y val (sin leakage).
# ─────────────────────────────────────────────────────────────────────────────

class GlobalPhysioNormalizer:
    """
    Normalización adaptada a las escalas reales de cada modalidad:
    - EEG: solo RobustScaler global (ya viene en escala razonable ~[-3, 3])
    - ET:  log1p en features de duración/tiempo + RobustScaler global
    - HR:  RobustScaler global (escala ya pequeña)
    """
    def __init__(self):
        self.scalers      = {}
        self.log_cols     = {}  # {modality: bool array} columnas que necesitan log
        self.clip_vals    = {}  # {modality: (lo, hi)} para outliers extremos
        self.fitted       = False

    # features de ET que están en nanosegundos o son conteos grandes
    ET_LOG_KEYWORDS = [
        "duration", "reaction_time", "count"
    ]

    def _needs_log(self, feature_names, modality):
        """Devuelve máscara booleana de qué columnas aplicar log1p."""
        if modality != "ET":
            return np.zeros(len(feature_names), dtype=bool)
        mask = np.array([
            any(kw in name.lower() for kw in self.ET_LOG_KEYWORDS)
            for name in feature_names
        ])
        return mask

    def _collect_all_vectors(self, dataset_list, modality):
        all_vecs = []
        for s in dataset_list:
            for vec in s["physio"].get(modality, []):
                if vec and not all(v == 0 for v in vec):
                    all_vecs.append(vec)
        return np.array(all_vecs, dtype=float) if all_vecs else None

    def _get_feature_names(self, dataset_list, modality):
        """Saca los nombres de features del primer sujeto que encuentre."""
        for s in dataset_list:
            raw = s.get("_physio_raw", {}).get(modality)
            if raw:
                return list(raw.keys())
        return []

    def fit(self, train_list):
        import warnings
        for modality in ["ET", "HR", "EEG"]:
            X = self._collect_all_vectors(train_list, modality)
            if X is None or X.shape[0] == 0:
                continue

            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

            # 1. Clip outliers extremos (solo para ET que tiene ns)
            pct = 2 if modality == "ET" else 1
            lo = np.percentile(X, pct, axis=0)
            hi = np.percentile(X, 100 - pct, axis=0)
            self.clip_vals[modality] = (lo, hi)
            X = np.clip(X, lo, hi)

            # 2. log1p solo en columnas de duración/conteo de ET
            #    (reaction_time en ms, durations en ns, counts)
            #    EEG y HR no necesitan transformación — ya están en escala razonable
            n_features = X.shape[1]
            if modality == "ET":
                # heurística: columnas con valores > 1000 probablemente son ns o ms
                col_max = np.max(np.abs(X), axis=0)
                log_mask = col_max > 1000
            else:
                log_mask = np.zeros(n_features, dtype=bool)

            self.log_cols[modality] = log_mask

            X_t = X.copy()
            if log_mask.any():
                # Asegurar positivo antes del log
                X_t[:, log_mask] = np.log1p(np.abs(X[:, log_mask]))

            X_t = np.nan_to_num(X_t, nan=0.0, posinf=0.0, neginf=0.0)

            # 3. RobustScaler global
            scaler = RobustScaler()
            scaler.fit(X_t)
            self.scalers[modality] = scaler

        self.fitted = True
        for mod in self.log_cols:
            n_log = self.log_cols[mod].sum()
            print(f"[GlobalPhysioNormalizer] {mod}: {n_log} columnas con log1p, "
                  f"{len(self.log_cols[mod]) - n_log} directas")

    def _transform_vec(self, vec, modality):
        lo, hi   = self.clip_vals[modality]
        log_mask = self.log_cols[modality]
        scaler   = self.scalers[modality]

        vec = np.array(vec, dtype=float)
        vec = np.nan_to_num(vec, nan=0.0, posinf=0.0, neginf=0.0)
        vec = np.clip(vec, lo, hi)

        if log_mask.any():
            vec[log_mask] = np.log1p(np.abs(vec[log_mask]))

        vec = np.nan_to_num(vec, nan=0.0, posinf=0.0, neginf=0.0)
        return scaler.transform(vec.reshape(1, -1)).flatten().tolist()

    def transform_dataset(self, dataset_list):
        assert self.fitted
        for sample in dataset_list:
            new_physio = {}
            for modality, subject_list in sample["physio"].items():
                if modality not in self.scalers or not subject_list:
                    new_physio[modality] = subject_list
                else:
                    new_physio[modality] = [
                        self._transform_vec(v, modality) for v in subject_list
                    ]
            sample["physio"] = new_physio
        return dataset_list

from joblib import Parallel, delayed

def _transform_sample(sample, normalizer):
    new_physio = {}
    for modality, subject_list in sample["physio"].items():
        if modality not in normalizer.scalers or not subject_list:
            new_physio[modality] = subject_list
        else:
            new_physio[modality] = [normalizer._transform_vec(v, modality) for v in subject_list]
    return {**sample, "physio": new_physio}

def transform_dataset_parallel(dataset_list, normalizer, n_jobs=-1):
    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=1)(
        delayed(_transform_sample)(s, normalizer) for s in dataset_list
    )
    return results

random.shuffle(dataset)

# from sklearn.model_selection import train_test_split
# from collections import Counter

# def majority_vote(votes):
#     counts = Counter(votes)
#     return counts.most_common(1)[0][0]

# labels_for_stratify = [majority_vote(x["label"]) for x in dataset]

# train_dataset, val_dataset = train_test_split(
#     dataset,
#     test_size=0.15,
#     stratify=labels_for_stratify,  # ajusta esto
#     random_state=42
# )
train_size    = int(len(dataset) * 0.85)
train_dataset = dataset[:train_size]
val_dataset   = dataset[train_size:]

normalizer = GlobalPhysioNormalizer()
normalizer.fit(train_dataset)
train_dataset = transform_dataset_parallel(train_dataset, normalizer)
val_dataset   = transform_dataset_parallel(val_dataset,   normalizer)
test_dataset = transform_dataset_parallel(test_dataset, normalizer)

os.makedirs("../data/processed", exist_ok=True)

with open("../data/processed/train.json", "w", encoding="utf-8") as f:
    json.dump(train_dataset, f, ensure_ascii=False, indent=2)
with open("../data/processed/val.json", "w", encoding="utf-8") as f:
    json.dump(val_dataset, f, ensure_ascii=False, indent=2)
with open("../data/processed/test.json", "w", encoding="utf-8") as f:
    json.dump(test_dataset, f, ensure_ascii=False, indent=2)
 
print(f"✅ test.json guardado → {len(test_dataset)} samples")

print(f"total: {len(dataset)} | train: {len(train_dataset)} | val: {len(val_dataset)}")


[GlobalPhysioNormalizer] ET: 12 columnas con log1p, 12 directas
[GlobalPhysioNormalizer] HR: 0 columnas con log1p, 4 directas
[GlobalPhysioNormalizer] EEG: 0 columnas con log1p, 80 directas


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done 1650 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done 1705 out of 1705 | elapsed:    1.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 301 out of 301 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 344 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 502 out of 502 | elapsed:    0.3s finished


✅ test.json guardado → 502 samples
total: 2006 | train: 1705 | val: 301


In [12]:
from collections import Counter

def majority_vote(labels):
    """
    labels: lista tipo ["YES", "NO", "YES", ...]
    """
    if not labels:
        return None
    
    labels = [l.strip().upper() for l in labels]
    return Counter(labels).most_common(1)[0][0]


def normalize_qwen_label(label: str):
    if not label:
        return None
    
    label = label.upper()
    
    if label.startswith("YES"):
        return "YES"
    elif label.startswith("NO"):
        return "NO"
    
    return None

In [13]:
correct = 0
total = 0
failed=0

y_true = []
y_pred = []

for meme_id, sample in data.items():
    
    # 🟢 ground truth (mayoritario)
    gt = majority_vote(sample["labels_task3_1"])
    
    # 🔵 predicción QWEN
    parsed = parse_qwen_verdict(sample.get("qwen_verdict_2", ""))
    pred = normalize_qwen_label(parsed["label"])
    
    if pred is None:
        failed+=1
        continue
    
    y_true.append(gt)
    y_pred.append(pred)
    
    if gt == pred:
        correct += 1
    
    total += 1

accuracy = correct / total if total > 0 else 0

print(f"Total evaluados: {total}")
print(f"Aciertos: {correct}")
print(f"Accuracy: {accuracy:.4f}")
print(failed)

Total evaluados: 0
Aciertos: 0
Accuracy: 0.0000
2006


In [14]:
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer

# =========================
# 🔧 CONFIG
# =========================
MODEL_NAME = "xlm-roberta-base"   # puedes cambiarlo si usas otro
MAX_LENGTH = 512

# =========================
# 🤖 Cargar tokenizer
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"✅ Tokenizer cargado: {MODEL_NAME}")
print(f"Max model length: {tokenizer.model_max_length}")

# =========================
# 🧠 Construcción de texto
# =========================
def build_text(sample, use_reasoning=True):
    parts = []

    if sample.get("qwen_ocr"):
        parts.append(f"[OCR] {sample['qwen_ocr']}")

    if sample.get("qwen_transcription"):
        parts.append(f"[TRANSCRIPTION] {sample['qwen_transcription']}")

    if use_reasoning and sample.get("qwen_reasoning"):
        parts.append(f"[REASONING] {sample['qwen_reasoning']}")

    return " </s> ".join(parts)

# =========================
# 📊 Análisis
# =========================
def analyze_token_lengths(dataset, tokenizer, max_length=256):
    lengths = []
    truncated = 0

    for sample in tqdm(dataset):
        text = build_text(sample)

        tokens = tokenizer(
            text,
            truncation=False,
            padding=False
        )["input_ids"]

        length = len(tokens)
        lengths.append(length)

        if length > max_length:
            truncated += 1

    lengths = np.array(lengths)

    print("\n📊 ===== TOKEN STATS =====")
    print(f"Samples: {len(lengths)}")
    print(f"Mean:    {np.mean(lengths):.2f}")
    print(f"Median:  {np.median(lengths):.2f}")
    print(f"Max:     {np.max(lengths)}")
    print(f"P90:     {np.percentile(lengths, 90):.2f}")
    print(f"P95:     {np.percentile(lengths, 95):.2f}")
    print(f"P99:     {np.percentile(lengths, 99):.2f}")

    print("\n⚠️ ===== TRUNCATION =====")
    print(f"Max length usado: {max_length}")
    print(f"Truncados: {truncated}/{len(lengths)} ({truncated/len(lengths):.2%})")

    # 🔎 Mostrar un ejemplo largo
    print("\n🔎 Ejemplo truncado:")
    for i, l in enumerate(lengths):
        if l > max_length:
            print(f"\n--- Sample {i} | tokens={l} ---")
            print(build_text(dataset[i])[:500])
            break


# =========================
# 🚀 EJECUCIÓN
# =========================
analyze_token_lengths(dataset, tokenizer, MAX_LENGTH)

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tokenizer cargado: xlm-roberta-base
Max model length: 512


100%|██████████| 2006/2006 [00:00<00:00, 5247.74it/s]


📊 ===== TOKEN STATS =====
Samples: 2006
Mean:    200.56
Median:  185.00
Max:     845
P90:     288.00
P95:     330.00
P99:     494.70

⚠️ ===== TRUNCATION =====
Max length usado: 512
Truncados: 17/2006 (0.85%)

🔎 Ejemplo truncado:

--- Sample 9 | tokens=531 ---
[OCR] Hi, I'm Magic Superstar Venus, your typical girl with a sarcastic sense of humor. My mom just told me that she had remarried. Her new husband and his son will be living with us. The son is Harry Styles, the most popular boy in school and my new stepbrother. Oh no, he walked in on me showering. I grab my towel. Now the lights went out because of a storm. I'm afraid of the thunder, but Harry sits with me and comforts me. Our parents just left for their honeymoon for 2 weeks and we are throwi
